In [ ]:
   import glob, os
   import numpy as np
   import pandas as pd

   DATA = os.path.expanduser("~/AppliedInvestments/data")
   OUT = f"{DATA}/parquet"

In [ ]:
import glob              
import pandas as pd

DATA = "../data"

# --- CRSP names file ---
names = pd.read_csv(glob.glob(f"{DATA}/honhxq*")[0])
names.columns = names.columns.str.lower()
start_col = "secinfostartdt" if "secinfostartdt" in names else "namedt"
end_col = "secinfoenddt" if "secinfoenddt" in names else "nameenddt"
names["ticker"] = names["ticker"].astype(str).str.upper().str.strip()
names[start_col] = pd.to_datetime(names[start_col], errors="coerce")
names[end_col] = pd.to_datetime(names[end_col], errors="coerce").fillna(pd.Timestamp("2100-01-01"))
names = names[["permno", "ticker", start_col, end_col]].drop_duplicates()

def match_permno(df, date_col, tol_days=0, ticker_col="ticker"):
    """Match ticker to PERMNO whose ticker window contains the date (+/- tolerance)."""
    d = df.reset_index(names="row_id").merge(
        names, left_on=ticker_col, right_on="ticker", how="left", suffixes=("", "_crsp"))
    lo = d[start_col] - pd.Timedelta(days=tol_days)
    hi = d[end_col] + pd.Timedelta(days=tol_days)
    d = d[(d[date_col] >= lo) & (d[date_col] <= hi)]
    d = d.sort_values(end_col).drop_duplicates("row_id", keep="last")  # latest window if ambiguous
    out = df.copy()
    out["permno"] = d.set_index("row_id")["permno"].reindex(out.index)
    return out

add = pd.read_csv(f"{DATA}/sp500_additions_2023_2025.csv", parse_dates=["effective_date"])
rem = pd.read_csv(f"{DATA}/sp500_removals_2023_2025.csv", parse_dates=["effective_date"])

# removed companies are often delisted just before the effective date -> small tolerance
add = match_permno(add, "effective_date", tol_days=10)
rem = match_permno(rem, "effective_date", tol_days=60)

print("Additions without PERMNO:\n", add.loc[add.permno.isna(), ["effective_date", "ticker", "company"]])
print("Removals without PERMNO:\n", rem.loc[rem.permno.isna(), ["effective_date", "ticker", "company"]])

Additions without PERMNO:
 Empty DataFrame
Columns: [effective_date, ticker, company]
Index: []
Removals without PERMNO:
 Empty DataFrame
Columns: [effective_date, ticker, company]
Index: []


In [7]:
old = pd.read_csv(f"{DATA}/sp500_constituent_changes.csv")
old["SP500 Start"] = pd.to_datetime(old["SP500 Start"])
old["SP500 End"] = pd.to_datetime(old["SP500 End"])
old = old.rename(columns={"PERMNO": "permno", "Ticker": "ticker", "Company": "company",
                          "SP500 Start": "start", "SP500 End": "end"})

LAST_OLD = old["end"].max()              # 2022-12-30: spells ending here are still open
END_NEW = pd.Timestamp("2025-12-31")     # end of our CRSP sample

# 1) close old open spells for companies removed 2023-2025, otherwise extend to END_NEW
rem_dates = rem.dropna(subset=["permno"]).groupby("permno")["effective_date"].min()
is_open = old["end"] >= LAST_OLD
old.loc[is_open, "end"] = (old.loc[is_open, "permno"].map(rem_dates)
                           .fillna(END_NEW).values)
old["source"] = "wrds_file"

# 2) new spells from additions (closed if removed later within the sample)
new = add.dropna(subset=["permno"]).copy()
new = new.rename(columns={"effective_date": "start"})
new["end"] = new["permno"].map(rem_dates).where(lambda s: s > new["start"]).fillna(END_NEW)
new["source"] = "manual_2023_2025"

cols = ["permno", "ticker", "company", "start", "end", "source"]
spells = pd.concat([old[cols], new[cols + [] ]], ignore_index=True)
spells["permno"] = spells["permno"].astype("Int64")
spells = spells.drop_duplicates(["permno", "start", "end"]).sort_values(["permno", "start"])

# 3) event table = additions with metadata (keep spin-offs flagged, filter later)
events = add.dropna(subset=["permno"]).rename(columns={"effective_date": "effective_date"})
old_add = old[old["start"] >= "1976-01-01"][["permno", "ticker", "company", "start"]].copy()
old_add = old_add.rename(columns={"start": "effective_date"})
old_add["event_type"] = "unknown"   # WRDS file has no event type: classify later
events = pd.concat([old_add, events[["permno", "ticker", "company", "effective_date", "event_type",
                                     "migration_from_midcap400", "announcement_date"]]],
                   ignore_index=True)

for name, df in [("sp500_spells", spells), ("sp500_events", events)]:
    try:
        df.to_parquet(f"{DATA}/{name}.parquet", index=False)
    except ImportError:
        df.to_csv(f"{DATA}/{name}.csv", index=False)

print(spells.groupby("source").size())
print(spells["end"].max(), events["effective_date"].dt.year.value_counts().sort_index().tail(5))

source
manual_2023_2025      49
wrds_file           1889
dtype: int64
2025-12-31 00:00:00 effective_date
2021    19
2022    17
2023    15
2024    16
2025    18
Name: count, dtype: int64


In [15]:
   import glob, os
   import pandas as pd

   DATA = os.path.expanduser("~/AppliedInvestments/data")
   OUT = f"{DATA}/parquet"
   os.makedirs(OUT, exist_ok=True)

   files = {
       "crsp_monthly":    "jhvzeg*",
       "crsp_daily":      "wxqvlg*",
       "crsp_delist":     "zopuwe*",
       "crsp_names":      "honhxq*",
       "index_monthly":   "pc8rgx*",
       "index_daily":     "z8pjwh*",
       "ccm_link":        "kg7lz7*",
       "comp_quarterly":  "pqwyvd*",
       "comp_annual":     "rdmfyn*",
   }
   date_cols = {"date", "datadate", "dlstdt", "linkdt", "linkenddt", "namedt", "nameenddt",
                "secinfostartdt", "secinfoenddt"}

   def clean(df):
       df.columns = df.columns.str.lower()
       for c in df.columns.intersection(date_cols):
           df[c] = pd.to_datetime(df[c], errors="coerce")
       return df

   for name, pattern in files.items():
       path = glob.glob(f"{DATA}/{pattern}")[0]
       target = f"{OUT}/{name}"
       os.makedirs(target, exist_ok=True)
       n = 0
       for i, chunk in enumerate(pd.read_csv(path, chunksize=3_000_000, low_memory=False)):
           chunk = clean(chunk)
           n += len(chunk)
           chunk.to_parquet(f"{target}/part_{i:03d}.parquet", index=False)
       print(f"{name}: {n:,} rows, {i + 1} parts")

crsp_monthly: 4,481,222 rows, 2 parts
crsp_daily: 92,586,611 rows, 31 parts
crsp_delist: 27,605 rows, 1 parts
crsp_names: 191,048 rows, 1 parts
index_monthly: 612 rows, 1 parts
index_daily: 12,858 rows, 1 parts
ccm_link: 32,824 rows, 1 parts
comp_quarterly: 2,035,812 rows, 1 parts
comp_annual: 583,112 rows, 1 parts


In [17]:
   m = pd.read_parquet(f"{OUT}/crsp_monthly")
   m["mthcaldt"] = pd.to_datetime(m["mthcaldt"])
   print(m["mthcaldt"].min(), m["mthcaldt"].max(), m["permno"].nunique(), "PERMNOs")
   print(m.isna().mean().round(3).to_string())
   print(m.head(3).to_string())

   nm = pd.read_parquet(f"{OUT}/crsp_names")
   print(nm.columns.tolist())
   print(nm.head(3).to_string())
   dl = pd.read_parquet(f"{OUT}/crsp_delist")
   print(dl.columns.tolist())

1975-01-31 00:00:00 2025-12-31 00:00:00 38272 PERMNOs
permno       0.000
hdrcusip     0.000
ticker       0.022
permco       0.000
mthcaldt     0.000
mthprc       0.016
mthdelflg    0.000
mthcap       0.017
mthret       0.017
mthretx      0.017
mthvol       0.070
   permno  hdrcusip ticker  permco   mthcaldt  mthprc mthdelflg   mthcap    mthret   mthretx    mthvol
0   10000  68391610  OMFGA    7952 1986-01-31  4.3750         N  16100.0  0.707317  0.707317  177082.0
1   10000  68391610  OMFGA    7952 1986-02-28  3.2500         N  11960.0 -0.257143 -0.257143   82800.0
2   10000  68391610  OMFGA    7952 1986-03-31  4.4375         N  16330.0  0.365385  0.365385  107801.0
['permno', 'permco', 'secinfostartdt', 'secinfoenddt', 'cusip', 'ticker', 'issuernm', 'usincflg', 'issuertype', 'securitytype', 'securitysubtype', 'sharetype', 'siccd', 'primaryexch']
   permno  permco secinfostartdt secinfoenddt     cusip ticker                   issuernm usincflg issuertype securitytype securitysubtype sh

In [18]:
   m = pd.read_parquet(f"{OUT}/crsp_monthly")
   m["mthcaldt"] = pd.to_datetime(m["mthcaldt"])
   nm = pd.read_parquet(f"{OUT}/crsp_names")
   for c in ["secinfostartdt", "secinfoenddt"]:
       nm[c] = pd.to_datetime(nm[c])

   # attach the security attributes valid at each month-end
   cols = ["permno", "secinfostartdt", "secinfoenddt", "issuertype", "securitytype",
           "securitysubtype", "sharetype", "usincflg", "primaryexch", "siccd"]
   x = pd.merge_asof(m.sort_values("mthcaldt"), nm[cols].sort_values("secinfostartdt"),
                     left_on="mthcaldt", right_on="secinfostartdt", by="permno")
   x = x[x["mthcaldt"] <= x["secinfoenddt"]]

   mask = (x.securitytype.eq("EQTY") & x.securitysubtype.eq("COM") & x.sharetype.eq("NS")
           & x.usincflg.eq("Y") & x.primaryexch.isin(["N", "A", "Q"]))
   panel = x[mask].drop(columns=["secinfostartdt", "secinfoenddt"]).sort_values(["permno", "mthcaldt"])

   print(len(m), "->", len(panel), "rows;", panel.permno.nunique(), "PERMNOs")
   print(panel.groupby(panel.mthcaldt.dt.year).permno.nunique().iloc[::5].to_string())
   print(panel[["mthret", "mthcap", "mthprc"]].describe().round(3).to_string())
   panel.to_parquet(f"{OUT}/panel_monthly.parquet", index=False)

4481222 -> 3167648 rows; 25136 PERMNOs
mthcaldt
1975    4989
1980    5043
1985    6325
1990    6291
1995    7656
2000    7337
2005    5279
2010    4365
2015    4172
2020    4095
2025    4110
            mthret        mthcap       mthprc
count  3155077.000  3.156171e+06  3156216.000
mean         0.013  2.760060e+06       46.467
std          0.193  2.847759e+07     2952.048
min         -1.000  1.056000e+01        0.006
25%         -0.067  2.411155e+04        4.650
50%          0.000  1.133306e+05       12.500
75%          0.073  6.605988e+05       25.625
max         26.584  4.921519e+09   800540.000
